# Train and evaluate a calibrated ModernBERT LLM router

ModernBERT predicts **whether a faster candidate preserves the quality of a strong fallback**. A deterministic analytical model—not ModernBERT and not candidate inference—estimates latency from model facts and prompt size.

The default run is a **random-split feasibility test**: can prompt content predict safe replacement at all? Repeated prompt content is grouped by a normalized SHA-256 hash, so the same question cannot cross train, validation, and test under different benchmark IDs. After random feasibility succeeds across several seeds, rerun with `SPLIT_MODE = "dataset_ood"` as a separate generalization stress test.

A successful sealed-test POC must activate the router, retain at least 98% quality at a one-sided 95% lower confidence bound, produce positive net analytical latency savings after router overhead, and route at least some prompts away from fallback. Validation uses an additional 1 percentage-point safety margin: it must reach a 99% LCB before the 98% sealed-test gate is opened.

> This proves feasibility under explicit analytical assumptions. It does not claim measured production latency.

## 1. One-cell Google Colab setup

Open this notebook in Google Colab, select a GPU runtime, and run every cell in order. This cell clones the `develop` branch, installs the project normally (not as an editable package), registers `src` in the live kernel, and verifies the import immediately. No terminal, runtime restart, or separate setup notebook is required.

In [ ]:
%cd /content
!test -d /content/LLM_Router || git clone --branch develop https://github.com/BrunoVitti96/LLM-router.git /content/LLM_Router
!git -C /content/LLM_Router pull --ff-only origin develop
%cd /content/LLM_Router
%pip install -q -U ".[notebook]"

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LLM_Router")
SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == "llm_router" or module_name.startswith("llm_router."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import llm_router

print(f"Router package ready from {Path(llm_router.__file__).resolve()}")

## 2. Experiment controls and automatic benchmark download

Start with `random`. Five stratified content-group folds produce an approximate 60/20/20 split while keeping repeated prompts together. Use `dataset_ood` only for a separate, harder artifact. For a serious conclusion, repeat both modes with at least seeds 42, 43, and 44. Never tune a later run from an already-opened test result.

The official archive is about 1.28 GB. The loader discovers its `dataset/split/model/file.json` structure instead of assuming a fixed wrapper name. No candidate LLM is installed or executed.

In [ ]:
import shutil
import tarfile
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display

from llm_router.config import DEFAULT_CONFIG
from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    benchmark_inventory,
    export_public_benchmark,
    load_llmrouterbench,
    make_complete_panel,
    run_public_benchmark,
    simulate_economics,
    split_benchmark,
)
from llm_router.utils.training import seed_everything

SEED = 42
SPLIT_MODE = "random"  # Use "dataset_ood" only for a separate stress test.
EPOCHS = 8  # Maximum; validation early stopping usually finishes sooner.
MINIMUM_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 2
VALIDATION_QUALITY_MARGIN = DEFAULT_CONFIG.validation_quality_margin
MINIMUM_MACRO_QUALITY_RETENTION = (
    DEFAULT_CONFIG.minimum_macro_quality_retention
)
MAXIMUM_QUALITY_LOSS_RATE_UCL = DEFAULT_CONFIG.maximum_quality_loss_rate_ucl
MINIMUM_ROUTED_SAFETY_PRECISION_LCB = (
    DEFAULT_CONFIG.minimum_routed_safety_precision_lcb
)
MINIMUM_GUARDED_DATASET_QUALITY_LCB = (
    DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
)
MINIMUM_GUARDED_DATASET_PROMPTS = (
    DEFAULT_CONFIG.minimum_guarded_dataset_prompts
)
CONSERVATIVE_ROUTER_OVERHEAD_S = (
    DEFAULT_CONFIG.conservative_router_overhead_s
)
ROUTER_CONFIG = replace(DEFAULT_CONFIG, seed=SEED)
DATA_ROOT = Path("/content/LLMRouterBench")
OUTPUT_DIR = Path(
    f"reports_benchmark/modernbert_hybrid_{SPLIT_MODE}_seed_{SEED}"
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

seed_everything(SEED)
inventory = benchmark_inventory(DATA_ROOT)
archive_preview = []
if inventory.empty:
    archive = hf_hub_download(
        repo_id="NPULH/LLMRouterBench",
        filename="bench-release.tar.gz",
        repo_type="dataset",
    )
    results_dir = DATA_ROOT / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as bundle:
        archive_preview = bundle.getnames()[:12]
        bundle.extractall(results_dir, filter="data")
    inventory = benchmark_inventory(DATA_ROOT)
if inventory.empty:
    extracted_preview = [
        str(path.relative_to(DATA_ROOT))
        for path in DATA_ROOT.rglob("*")
        if path.is_file()
    ][:20]
    raise RuntimeError(
        "No dataset/split/model/*.json benchmark layout was found after "
        f"extraction. Archive entries: {archive_preview}. "
        f"Extracted files: {extracted_preview}"
    )
DATA_ROOT = Path(inventory.iloc[0]["file"]).parents[3]
print({
        "device": DEVICE,
        "split_mode": SPLIT_MODE,
        "seed": SEED,
        "data_root": str(DATA_ROOT.resolve()),
        "output_dir": str(OUTPUT_DIR),
    })
if DEVICE == "cpu":
    print("Warning: CPU training works for a smoke test but will be slow. A GPU is recommended.")

## 3. Inspect available datasets and models

Do this before editing model profiles. The strings in `SELECTED_MODELS` must exactly match model directory names shown below. Choose at least two candidates and at least three datasets for a dataset-disjoint split.

In [ ]:
inventory = benchmark_inventory(DATA_ROOT)
assert not inventory.empty, "No LLMRouterBench result files were found."
inventory_summary = (
    inventory.groupby(["model", "dataset", "source_split"])
    .size()
    .rename("files")
    .reset_index()
)
display(inventory_summary)
print(f"Available models: {inventory.model.nunique()}")
print(f"Available datasets: {inventory.dataset.nunique()}")

## 4. Declare a non-dominated candidate panel

The previous run included NVIDIA-Nemotron-Nano-9B-v2, but it was analytically slower than the Qwen fallback and had a 0% oracle-selection rate. It could never change the latency-minimizing decision.

The clean default is therefore:

- `Fin-R1`: approximately 7B parameters, the faster replacement candidate;
- `Qwen3-8B`: 8.2B parameters, the potential quality fallback.

Both generate autoregressively at BF16 precision. The public pool contains no diffusion model; add one only when matching pre-collected quality results and sourced denoising settings exist.

In [ ]:
MODEL_PROFILES = {
    "Fin-R1": {
        "parameters_billions": 7.0,
        "active_parameters_billions": 7.0,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
    "Qwen3-8B": {
        "parameters_billions": 8.2,
        "active_parameters_billions": 8.2,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
}
SELECTED_MODELS = tuple(MODEL_PROFILES)
assert len(SELECTED_MODELS) >= 2, "Routing requires at least two models."
missing_models = sorted(set(SELECTED_MODELS) - set(inventory.model))
assert not missing_models, (
    f"Configured models were not found: {missing_models}. "
    f"Available models: {sorted(inventory.model.unique())}"
)
print(SELECTED_MODELS)

## 5. Load pre-collected quality and calculate analytical latency

The benchmark provides candidate answers and quality scores. `simulate_economics` calculates latency without loading those candidates. Expected output length is a bounded function of prompt length, so the policy cannot peek at a candidate's realized response length. The panel also records a normalized prompt hash and duplicate-group size for content-level leakage auditing.

In [ ]:
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        **settings,
    )
    for name, settings in MODEL_PROFILES.items()
)
scenario = EconomicsScenario(
    name="notebook-analytical-latency-poc",
    as_of=pd.Timestamp.utcnow().date().isoformat(),
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="POC assumptions; not measured production latency.",
)

records = load_llmrouterbench(DATA_ROOT, models=SELECTED_MODELS)
simulated = simulate_economics(records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)
assert simulated.latency_source.eq("analytical").all()
print({"complete_prompts": len(panel.examples), "models": panel.models})
print({
    "repeated_prompt_rows": int((panel.examples.prompt_group_size > 1).sum()),
    "largest_prompt_group": int(panel.examples.prompt_group_size.max()),
})
display(
    simulated.groupby("model")["simulated_latency_s"]
    .agg(["min", "median", "mean", "max"])
    .sort_values("mean")
)

### Leakage check

This deliberately changes every realized completion length. Analytical latency must remain identical because only prompt size and declared model/scenario properties are allowed to affect it.

In [ ]:
counterfactual_records = records.copy()
counterfactual_records["completion_tokens"] *= 100
counterfactual = simulate_economics(counterfactual_records, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual.simulated_latency_s,
), "Analytical latency unexpectedly depends on realized completion length."
print("Passed: candidate completion tokens do not affect analytical latency.")

## 6. Freeze train, validation, and sealed-test splits

The fallback is selected from training quality only. Validation is used for checkpoint selection, out-of-fold calibration, threshold selection, and the activation gate. Test outcomes remain sealed until the complete policy is frozen.

- `random`: normalized-prompt-hash-disjoint feasibility test; start here.
- `dataset_ood`: dataset-disjoint generalization stress test; run separately.

In [ ]:
split = split_benchmark(panel, mode=SPLIT_MODE, seed=SEED)
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "prompts": [len(split.train), len(split.validation), len(split.test)],
        "datasets": [
            split.train_datasets,
            split.validation_datasets,
            split.test_datasets,
        ],
    }
)
display(split_summary)
assert not set(split.train).intersection(split.validation)
assert not set(split.train).intersection(split.test)
assert not set(split.validation).intersection(split.test)
split_hashes = {
    name: set(panel.examples.iloc[indices].prompt_hash)
    for name, indices in {
        "train": split.train,
        "validation": split.validation,
        "test": split.test,
    }.items()
}
assert split_hashes["train"].isdisjoint(split_hashes["validation"])
assert split_hashes["train"].isdisjoint(split_hashes["test"])
assert split_hashes["validation"].isdisjoint(split_hashes["test"])
if SPLIT_MODE == "dataset_ood":
    assert set(split.train_datasets).isdisjoint(split.validation_datasets)
    assert set(split.train_datasets).isdisjoint(split.test_datasets)
    assert set(split.validation_datasets).isdisjoint(split.test_datasets)

## 7. Verify validation-only oracle headroom and scenario sensitivity

The hindsight oracle sees recorded outcomes and chooses the fastest model that matches or beats fallback quality. It is unattainable at deployment, but it proves whether routing opportunity exists.

This cell uses **validation only** and repeats the calculation under conservative compute, conservative bandwidth, longer-output, and larger fixed-overhead assumptions. If modest changes erase headroom, the latency claim is too fragile to train. Router-overhead sensitivity is evaluated later on the frozen test policy because it depends on how often the router selects the faster model.

In [ ]:
def validation_oracle_metrics(candidate_panel):
    fallback_index = int(candidate_panel.score[split.train].mean(axis=0).argmax())
    target = oracle_choices(
        candidate_panel.score,
        candidate_panel.latency,
        fallback_index=fallback_index,
    )
    indices = split.validation
    rows = np.arange(len(indices))
    chosen = target[indices]
    fallback_quality = candidate_panel.score[indices, fallback_index].mean()
    oracle_quality = candidate_panel.score[indices][rows, chosen].mean()
    fallback_latency = candidate_panel.latency[indices, fallback_index].mean()
    oracle_latency = candidate_panel.latency[indices][rows, chosen].mean()
    return {
        "fallback_model": candidate_panel.models[fallback_index],
        "quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "latency_savings": 1 - oracle_latency / fallback_latency,
        "fallback_usage": np.mean(chosen == fallback_index),
    }

sensitivity_settings = {
    "balanced": {},
    "compute_conservative": {"effective_tflops": 40.0},
    "bandwidth_conservative": {"memory_bandwidth_gbps": 600.0},
    "larger_fixed_overhead": {"fixed_model_overhead_s": 0.050},
    "longer_outputs": {
        "output_base_tokens": 48.0,
        "output_tokens_per_prompt_token": 0.35,
    },
}
sensitivity_rows = []
for name, overrides in sensitivity_settings.items():
    variant = replace(scenario, name=name, **overrides)
    variant_records = simulate_economics(records, variant)
    variant_panel = make_complete_panel(variant_records, models=SELECTED_MODELS)
    assert variant_panel.examples.example_id.equals(panel.examples.example_id)
    sensitivity_rows.append(
        {"scenario": name, **validation_oracle_metrics(variant_panel)}
    )
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
assert (sensitivity.latency_savings > 0).all(), (
    "Oracle headroom is not robust across the declared sensitivity scenarios."
)

## 8. Understand the objective, loss, and calibration

For each non-fallback candidate, replacement safety is:

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_f(x)-\epsilon_q].$$

The deployable head now uses **class-balanced** binary cross-entropy. Candidate $m$ receives positive weight $N_{unsafe}/N_{safe}$, clipped for numerical stability. This prevents a high safe-rate prior from winning without learning prompt differences.

A second, training-only head imitates the hindsight oracle and penalizes expected quality risk plus latency regret:

$$\mathcal L_{oracle}=(1+g_o)CE(z,o)+4\sum_m p_m d_m+\sum_m p_m r_m.$$

The complete objective is:

$$\boxed{\mathcal L_{train}=\mathcal L_{balanced\ safety}+0.25\mathcal L_{oracle}}.$$

After checkpoint selection, each safety logit receives per-candidate Platt calibration. Threshold search uses **out-of-fold validation probabilities**, so a validation row never calibrates itself. The exported deployment scaler is fitted on all validation rows. ModernBERT never predicts latency or the final model.

## 9. Train ModernBERT

Exact dataset names are excluded from router inputs. ModernBERT sees prompt text and prompt-token count, reducing brittle dataset-identity memorization. The LoRA adapter uses a lower learning rate than the newly initialized heads, and validation early stopping prevents the falling training loss from hiding overfitting. On FP16 GPUs, skipped optimizer steps do not advance the learning-rate scheduler.

The input diagnostics use ModernBERT's own tokenizer. For example, if `truncation_rate=0.12`, then 12% of prompts exceeded the 512-token router limit; that would motivate a longer limit or chunked encoding in a later experiment.

In [ ]:
training = train_modernbert_hybrid_poc(
    panel,
    split,
    config=ROUTER_CONFIG,
    epochs=EPOCHS,
    batch_size=8,
    learning_rate=1e-4,
    head_learning_rate=2e-4,
    minimum_epochs=MINIMUM_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    quality_epsilon=0.0,
    safety_loss_weight=1.0,
    oracle_auxiliary_weight=0.25,
    device=DEVICE,
)
display(training.history)
display(training.calibration_diagnostics)
display(pd.DataFrame([training.input_diagnostics]))
print({
    "best_epoch": training.best_epoch,
    "epochs_completed": training.epochs_completed,
    "stopped_early": training.stopped_early,
    "training_seconds": round(training.training_seconds, 1),
})
assert training.safety_probabilities.shape == panel.score.shape
assert np.allclose(
    training.safety_probabilities[:, training.fallback_index], 1.0
)
assert np.all(
    (training.safety_probabilities >= 0)
    & (training.safety_probabilities <= 1)
)

## 10. Freeze the policy, then open the sealed test

Validation selects the confidence threshold from a dense, predeclared grid and may disable the router. Selection requires `98% + 1% margin = 99%` aggregate validation LCB, at least 98% macro-dataset LCB, at most a 2.5% quality-loss-rate UCL, at least a 90% routed-safety-precision LCB, at least a 90% worst-dataset LCB among datasets with 100 or more prompts, and positive analytical savings at both 4 ms and a conservative 20 ms router overhead. Only after that policy is frozen does the report evaluate test outcomes against the corresponding sealed-test gates.

Numerical example: 92% routed safety precision can look adequate, but if its one-sided 95% lower bound is only 89%, the policy remains disabled. Likewise, 2.3% savings at 4 ms do not qualify if they turn negative at 20 ms.

In [ ]:
result = run_public_benchmark(
    panel,
    split,
    objective="latency",
    minimum_quality_retention=0.98,
    confidence=0.95,
    validation_quality_margin=VALIDATION_QUALITY_MARGIN,
    minimum_predicted_savings=0.02,
    router_overhead_s=scenario.router_overhead_s,
    conservative_router_overhead_s=CONSERVATIVE_ROUTER_OVERHEAD_S,
    minimum_macro_quality_retention=MINIMUM_MACRO_QUALITY_RETENTION,
    maximum_quality_loss_rate_ucl=MAXIMUM_QUALITY_LOSS_RATE_UCL,
    minimum_routed_safety_precision_lcb=(
        MINIMUM_ROUTED_SAFETY_PRECISION_LCB
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        MINIMUM_GUARDED_DATASET_QUALITY_LCB
    ),
    minimum_guarded_dataset_prompts=MINIMUM_GUARDED_DATASET_PROMPTS,
    seed=SEED,
    routing_probabilities=training.safety_probabilities,
    router_name="modernbert_hybrid_router",
    decision_metadata={
        "router_input_tokens": training.router_input_lengths,
        "router_was_truncated": training.router_was_truncated,
    },
)
assert result.fallback_model == panel.models[training.fallback_index]
display(result.threshold_search)
display(result.summary)
display(result.candidate_diagnostics)
router_dataset_metrics = result.per_dataset_metrics.loc[
    result.per_dataset_metrics.strategy.eq("modernbert_hybrid_router")
].sort_values("quality_retention")
display(router_dataset_metrics)
overhead_sensitivity = result.router_overhead_sensitivity
display(overhead_sensitivity)
print({
    "break_even_router_overhead_ms": round(
        overhead_sensitivity.break_even_router_overhead_ms.iloc[0], 2
    )
})
print(
    {
        "poc_passed": result.poc_passed,
        "router_active": result.router_active,
        "selected_threshold": result.selected_threshold,
        "fallback_model": result.fallback_model,
        "failure_reasons": result.failure_reasons,
    }
)

### Interpret the result honestly

- **Oracle savings near zero:** the panel or assumptions offer no useful headroom.
- **Oracle headroom but `router_active=False`:** ModernBERT found no validation-safe policy.
- **Router active but `poc_passed=False`:** validation looked promising, but sealed-test behavior did not generalize.
- **Random passes and dataset-OOD fails:** feasibility exists, but cross-domain generalization does not.
- **Both modes pass across several seeds and sensitivity scenarios:** this is a credible analytical-latency POC.

A fallback-only outcome proves the safety guard worked; it does not prove the router worked.

## 11. Export the complete report and router artifact

The export contains strategy metrics, threshold search, sealed-test decisions, full per-candidate safety probabilities, candidate and per-dataset diagnostics, harm-rate and routed-precision confidence bounds, safe-opportunity recall, analytical and router-overhead sensitivity, per-prompt input-truncation diagnostics, explicit pass/fail reasons, training history, calibration and ranking diagnostics, Platt parameters, LoRA weights, both heads, and tokenizer.

Numerical example: a 2% pooled saving can still be unacceptable if one small dataset retains only 90% quality, or if a measured 50 ms router overhead exceeds the reported break-even overhead. The additional CSVs make both problems visible.

In [ ]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
sensitivity.to_csv(report_dir / "validation_sensitivity.csv", index=False)
artifact_dir = export_modernbert_hybrid_poc(
    training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    poc_passed=result.poc_passed,
    failure_reasons=result.failure_reasons,
    minimum_predicted_savings=0.02,
    validation_quality_margin=VALIDATION_QUALITY_MARGIN,
    minimum_macro_quality_retention=MINIMUM_MACRO_QUALITY_RETENTION,
    maximum_quality_loss_rate_ucl=MAXIMUM_QUALITY_LOSS_RATE_UCL,
    minimum_routed_safety_precision_lcb=(
        MINIMUM_ROUTED_SAFETY_PRECISION_LCB
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        MINIMUM_GUARDED_DATASET_QUALITY_LCB
    ),
    minimum_guarded_dataset_prompts=MINIMUM_GUARDED_DATASET_PROMPTS,
    conservative_router_overhead_s=CONSERVATIVE_ROUTER_OVERHEAD_S,
    config=ROUTER_CONFIG,
)
print({"reports": str(report_dir.resolve()), "artifact": str(artifact_dir.resolve())})

## 12. Download the result before Colab shuts down

The ZIP path contains split mode and seed, so feasibility and dataset-OOD artifacts cannot silently overwrite one another.

In [ ]:
bundle_path = shutil.make_archive(
    str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR
)
print(f"Created {bundle_path}")
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    print("Not running in Colab; download the ZIP from the printed path.")

## 13. Final POC checklist

Before presenting a result, confirm:

- random feasibility passed across at least three seeds;
- dataset-OOD was run as a separate, clearly labeled stress-test artifact;
- every retained alternative has a non-zero oracle-selection rate;
- normalized prompt hashes are disjoint across random train, validation, and test;
- calibration improved or did not materially worsen Brier score, Brier skill, ECE, unsafe average precision, and AUROC;
- macro retention, guarded-dataset retention, harm-rate UCL, routed-safety-precision LCB, and safe-opportunity recall are acceptable;
- ModernBERT input truncation is reported and investigated if material;
- no candidate inference or realized completion length was used for latency;
- fallback selection used training quality only;
- checkpointing, calibration, threshold selection, and activation used validation only;
- test outcomes were opened only after policy freeze;
- net savings include router overhead and remain positive across plausible overhead values;
- conclusions survive all declared sensitivity scenarios; and
- every latency claim says **analytical** or **simulated**, never measured.